# 创建 Collection（集合）

## 用途
教学演示 - 如何在 Milvus 中创建集合

## 核心概念
- Collection = 数据库中的"表"
- Field Schema（字段模式）定义
- 向量字段 vs 标量字段

## 运行前检查
1. 已安装依赖：`pip install -r requirements.txt`
2. 已完成 `01_connect_milvus.py` 的学习

In [1]:
# 导入 Milvus 配置（兼容 Jupyter Notebook）

from rag_examples.milvus_config import MILVUS_URI

print(f"Milvus 连接地址：{MILVUS_URI}")

Milvus 连接地址：http://47.115.57.130:19530


## 第一部分：理解 Collection

### 什么是 Collection？

**定义**
- Collection = 集合 = 数据库中的"表"
- 用于存储向量数据和关联的标量数据

### 生活化比喻

Collection = "Excel 表格"

| ID  | 文本  |   向量      |  类别 |
|-----|-------|-------------|-------|
|  1  | 苹果  | [0.1,..]    |  水果 |
|  2  | 香蕉  | [0.2,..]    |  水果 |
|  3  | iPhone| [0.3,..]    |  数码 |

### 字段类型
- 向量字段（Vector Field）：存储 Embedding 生成的向量
- 标量字段（Scalar Field）：存储 ID、文本、数字等传统数据

### 主键说明
- Milvus 需要指定一个主键字段（Primary Key）
- 主键可以是自增的（Auto ID）或手动指定

In [18]:
def create_simple_collection():
    """
    创建一个简单的 Collection

    包含最基础的字段：ID、文本、向量
    """
    print("=" * 60)
    print("示例 1: 创建简单的 Collection")
    print("=" * 60)

    from pymilvus import MilvusClient, DataType


    # 连接 Milvus
    client = MilvusClient(uri=MILVUS_URI, db_name="ai80")

    # Collection 名称
    collection_name = "simple_docs"

    # 建表前先删表（确保可重复运行）
    if client.has_collection(collection_name):
        print(f"删除已存在的集合：{collection_name}")
        client.drop_collection(collection_name)

    print(f"创建集合：{collection_name}")

    # 创建 Collection
    # auto_id=True 表示 ID 自动生成，无需手动指定
    client.create_collection(
        collection_name=collection_name,
        dimension=768,           # 向量维度，需与 Embedding 模型匹配
        auto_id=True,            # 主键自增
        metric_type="COSINE",    # 相似度度量类型：COSINE/L2/IP
    )

    print("✓ 创建成功！")
    print()

    # 查看 Collection 信息
    info = client.describe_collection(collection_name)
    print(f"集合信息：")
    print(f"  名称：{info['collection_name']}")
    # 从 fields 中获取维度
    dimension = None
    for field in info.get('fields', []):
        if field.get('type') == 101:  # 101 表示向量类型
            dimension = field.get('params', {}).get('dim')
            break
    print(f"  向量维度：{dimension}")
    print(f"  度量类型：COSINE")
    print(f"  主键自增：{info['auto_id']}")

    return client, collection_name

In [19]:
# 运行示例 1
client, collection_name = create_simple_collection()

示例 1: 创建简单的 Collection
创建集合：simple_docs
✓ 创建成功！

集合信息：
  名称：simple_docs
  向量维度：768
  度量类型：COSINE
  主键自增：True


## 示例 2: 自定义字段的 Collection

创建自定义字段的 Collection，定义多个标量字段，支持更复杂的查询。

In [2]:
def create_custom_collection():
    """
    创建自定义字段的 Collection

    定义多个标量字段，支持更复杂的查询
    """
    print("=" * 60)
    print("示例 2: 创建自定义字段的 Collection")
    print("=" * 60)

    from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType

    # 连接 Milvus
    client = MilvusClient(uri=MILVUS_URI)

    # Collection 名称
    collection_name = "custom_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        print(f"删除已存在的集合：{collection_name}")
        client.drop_collection(collection_name)

    print(f"创建集合：{collection_name}")

    # 定义字段模式
    # FieldSchema 定义每个字段的属性
    fields = [
        # 主键字段：自增 INT64
        # 一个FieldSchema 对象表示一个字段
        FieldSchema(
            name="id",
            dtype=DataType.INT64,
            is_primary=True,      # 是主键
            auto_id=True          # 自动生成
        ),
        # 文本内容字段
        FieldSchema(
            name="content",
            dtype=DataType.VARCHAR,
            max_length=65535      # 最大长度
        ),
        # 标题字段
        FieldSchema(
            name="title",
            dtype=DataType.VARCHAR,
            max_length=256
        ),
        # 类别字段（用于筛选）
        FieldSchema(
            name="category",
            dtype=DataType.VARCHAR,
            max_length=64
        ),
        # 浏览量字段（用于排序）
        FieldSchema(
            name="views",
            dtype=DataType.INT64
        ),
        # 向量字段
        FieldSchema(
            name="embedding",
            dtype=DataType.FLOAT_VECTOR,  # 浮点向量
            dim=768                       # 向量维度
        ),
    ]

    # 创建 Schema
    schema = CollectionSchema(fields=fields)

    # 创建 Collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        metric_type="COSINE"
    )

    print("✓ 创建成功！")
    print()

    # 查看字段信息
    info = client.describe_collection(collection_name)
    print(f"字段列表：")
    for field in info['fields']:
        field_name = field['name']
        field_type = field['type']
        is_primary = field.get('is_primary', False)
        print(f"  - {field_name}: {field_type} {'(主键)' if is_primary else ''}")

In [3]:
# 运行示例 2
create_custom_collection()

示例 2: 创建自定义字段的 Collection
创建集合：custom_docs
✓ 创建成功！

字段列表：
  - id: 5 (主键)
  - content: 21 
  - title: 21 
  - category: 21 
  - views: 5 
  - embedding: 101 


## 示例 3: 多向量字段 Collection

创建包含多个向量字段的 Collection，适用于多模态场景（如同时有文本向量和图像向量）。

In [12]:
def create_multi_vector_collection():
    """
    创建包含多个向量字段的 Collection

    适用于多模态场景（如同时有文本向量和图像向量）
    """
    print("=" * 60)
    print("示例 3: 创建多向量字段的 Collection")
    print("=" * 60)

    from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "multi_vector_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    print(f"创建集合：{collection_name}")

    # 定义字段
    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=65535),

        # 文本向量（768 维）
        FieldSchema(
            name="text_embedding",
            dtype=DataType.FLOAT_VECTOR,
            dim=768
        ),

        # 图像向量（512 维）
        # 适用于多模态检索场景
        FieldSchema(
            name="image_embedding",
            dtype=DataType.FLOAT_VECTOR,
            dim=512
        ),
    ]

    schema = CollectionSchema(fields=fields)

    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        metric_type="COSINE"
    )

    print("✓ 创建成功！")
    print()
    print("💡 多向量字段适用于：")
    print("   - 图文混合检索")
    print("   - 多模态 RAG 系统")
    print("   - 跨模态搜索（以文搜图、以图搜文）")

In [13]:
# 运行示例 3
create_multi_vector_collection()

示例 3: 创建多向量字段的 Collection
创建集合：multi_vector_docs
✓ 创建成功！

💡 多向量字段适用于：
   - 图文混合检索
   - 多模态 RAG 系统
   - 跨模态搜索（以文搜图、以图搜文）


## 示例 4: 动态字段（Dynamic Field）- Milvus 2.3+ 新特性

创建启用动态字段的 Collection。

### 什么是动态字段？

动态字段是 Milvus 2.3 引入的新特性，允许：
1. 插入数据时自动添加未预定义的字段
2. 无需修改 Schema 就能存储不同结构的数据
3. 类似 MongoDB 的"文档型"体验

### 使用场景：
- 文档元数据不固定（如不同来源的文档有不同属性）
- 快速原型开发（不需要一开始就定义所有字段）
- 多租户系统（不同租户需要不同的字段）

### 注意事项：
- 动态字段会增加少量存储开销
- 查询时才能知道动态字段的值
- 建议只用于标量字段，向量字段必须预定义

In [14]:
def create_dynamic_field_collection():
    """
    创建启用动态字段的 Collection

    Milvus 2.3+ 新特性：动态字段
    参考：https://milvus.io/docs/zh/enable-dynamic-field.md

    动态字段的优势：
    - 无需预先定义所有字段
    - 插入数据时可以动态添加新字段
    - 适合字段不固定或频繁变化的场景
    """
    print("=" * 60)
    print("示例 4: 动态字段（Milvus 2.3+ 新特性）")
    print("=" * 60)

    from pymilvus import MilvusClient, FieldSchema, DataType, CollectionSchema
    from pymilvus.milvus_client import IndexParams

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "dynamic_field_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 定义字段（只定义核心字段）
    fields = [
        # 主键
        FieldSchema(
            name="id",
            dtype=DataType.INT64,
            is_primary=True,
            auto_id=True
        ),
        # 内容
        FieldSchema(
            name="content",
            dtype=DataType.VARCHAR,
            max_length=65535
        ),
        # 向量字段（必须预定义）
        FieldSchema(
            name="vector",
            dtype=DataType.FLOAT_VECTOR,
            dim=768
        ),
    ]

    # 关键：启用动态字段
    # enable_dynamic_field=True 后，插入的数据可以包含额外的字段
    schema = CollectionSchema(
        fields=fields,
        enable_dynamic_field=True  # 启用动态字段
    )

    print("创建集合（启用动态字段）...")
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        metric_type="COSINE"
    )

    print("✓ 创建成功！")
    print()

    # 演示：插入带有动态字段的数据
    print("插入数据（包含未预定义的动态字段）：")

    # 这些数据包含了 Schema 中未定义的字段：author, tags, publish_date
    data = [
        {
            "content": "人工智能简介",
            "vector": [0.1] * 768,
            # 以下字段未在 Schema 中定义，是动态字段
            "author": "张三",
            "tags": ["AI", "科技"],
            "publish_date": "2024-01-15"
        },
        {
            "content": "机器学习基础",
            "vector": [0.2] * 768,
            # 这条数据的动态字段又不同
            "author": "李四",
            "course_id": 101,
            "difficulty": "中级"
        },
        {
            "content": "深度学习进阶",
            "vector": [0.3] * 768,
            # 另一组动态字段
            "author": "王五",
            "video_url": "https://example.com/video",
            "duration_minutes": 45
        },
    ]

    client.insert(collection_name=collection_name, data=data)
    print(f"  插入 {len(data)} 条数据，每条数据有不同的动态字段")
    print()

    # 查询验证
    print("创建索引并加载集合...")
    index_params = IndexParams()
    index_params.add_index(field_name="vector", index_type="FLAT", metric_type="COSINE")
    client.create_index(collection_name=collection_name, index_params=index_params)
    client.load_collection(collection_name=collection_name)

    print("查询所有数据，查看动态字段：")
    results = client.query(
        collection_name=collection_name,
        filter="",
        output_fields=["*"],  # 使用 * 获取所有字段（包括动态字段）
        limit=10
    )

    for i, item in enumerate(results):
        print(f"\n  [{i+1}] ID: {item.get('id')}")
        print(f"      内容：{item.get('content', '')[:20]}...")
        # 动态字段也会被返回
        dynamic_keys = [k for k in item.keys() if k not in ['id', 'content', 'vector']]
        if dynamic_keys:
            print(f"      动态字段：{dynamic_keys}")
            for key in dynamic_keys:
                print(f"        - {key}: {item[key]}")

    print("\n[OK] 动态字段演示完成！")

    return client, collection_name

In [15]:
# 运行示例 4
create_dynamic_field_collection()

示例 4: 动态字段（Milvus 2.3+ 新特性）
创建集合（启用动态字段）...
✓ 创建成功！

插入数据（包含未预定义的动态字段）：
  插入 3 条数据，每条数据有不同的动态字段

创建索引并加载集合...
查询所有数据，查看动态字段：

  [1] ID: 465920187023367922
      内容：人工智能简介...
      动态字段：['author', 'tags', 'publish_date']
        - author: 张三
        - tags: ['AI', '科技']
        - publish_date: 2024-01-15

  [2] ID: 465920187023367923
      内容：机器学习基础...
      动态字段：['author', 'course_id', 'difficulty']
        - author: 李四
        - course_id: 101
        - difficulty: 中级

  [3] ID: 465920187023367924
      内容：深度学习进阶...
      动态字段：['author', 'video_url', 'duration_minutes']
        - author: 王五
        - video_url: https://example.com/video
        - duration_minutes: 45

[OK] 动态字段演示完成！


(<pymilvus.milvus_client.milvus_client.MilvusClient at 0x1e905bc92e0>,
 'dynamic_field_docs')

## 示例 5: 度量类型详解

| 类型 | 计算方式 | 特点 |
|------|----------|------|
| L2 (EUCLIDEAN) | 欧几里得距离 √(Σ(xi-yi)²) | 值越小越相似 |
| IP (INNER_PRODUCT) | 内积 Σ(xi*yi) | 值越大越相似 |
| COSINE | 余弦相似度 cos(θ) | 值越大越相似 (范围 -1~1) |

### 如何选择度量类型？

**1. COSINE（推荐默认选择）**
- 最常用，适用于大多数语义检索场景
- 不受向量长度影响，只关注方向
- Embedding 模型通常输出归一化向量

**2. L2（欧几里得距离）**
- 适用于向量未归一化的场景
- 考虑向量的绝对位置

**3. IP（内积）**
- 与 COSINE 在归一化后等价
- 某些场景下计算更快

In [16]:
def metric_types_explained():
    """
    详解 Milvus 支持的度量类型

    不同度量类型适用于不同场景
    """
    print("=" * 60)
    print("示例 5: 度量类型详解")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ Milvus 度量类型对比                                      │
├──────────────┬────────────────┬─────────────────────────┤
│    类型       │    计算方式     │         特点             │
├──────────────┼────────────────┼─────────────────────────┤
│ L2           │ 欧几里得距离    │ 值越小越相似             │
│ (EUCLIDEAN)  │ √(Σ(xi-yi)²)   │ 适用于归一化前的向量     │
├──────────────┼────────────────┼─────────────────────────┤
│ IP           │ 内积            │ 值越大越相似             │
│ (INNER_PRODUCT)│ Σ(xi*yi)     │ 与 COSINE 类似           │
├──────────────┼────────────────┼─────────────────────────┤
│ COSINE       │ 余弦相似度      │ 值越大越相似 (范围 -1~1) │
│              │ cos(θ)         │ 最常用的类型             │
└──────────────┴────────────────┴─────────────────────────┘

💡 如何选择度量类型？

1. COSINE（推荐默认选择）
   ✓ 最常用，适用于大多数语义检索场景
   ✓ 不受向量长度影响，只关注方向
   ✓ Embedding 模型通常输出归一化向量

2. L2（欧几里得距离）
   ✓ 适用于向量未归一化的场景
   ✓ 考虑向量的绝对位置

3. IP（内积）
   ✓ 与 COSINE 在归一化后等价
   ✓ 某些场景下计算更快

📊 示例对比:

假设有两个向量:
A = [1, 2, 3]
B = [2, 4, 6]  (A 的 2 倍)

- COSINE(A, B) = 1.0  (方向相同，完全相似)
- L2(A, B) = √14 ≈ 3.74 (距离不为 0)

结论：COSINE 更关注"语义方向"，L2 更关注"绝对位置"
""")

In [17]:
# 运行示例 5
metric_types_explained()

示例 5: 度量类型详解

┌─────────────────────────────────────────────────────────┐
│ Milvus 度量类型对比                                      │
├──────────────┬────────────────┬─────────────────────────┤
│    类型       │    计算方式     │         特点             │
├──────────────┼────────────────┼─────────────────────────┤
│ L2           │ 欧几里得距离    │ 值越小越相似             │
│ (EUCLIDEAN)  │ √(Σ(xi-yi)²)   │ 适用于归一化前的向量     │
├──────────────┼────────────────┼─────────────────────────┤
│ IP           │ 内积            │ 值越大越相似             │
│ (INNER_PRODUCT)│ Σ(xi*yi)     │ 与 COSINE 类似           │
├──────────────┼────────────────┼─────────────────────────┤
│ COSINE       │ 余弦相似度      │ 值越大越相似 (范围 -1~1) │
│              │ cos(θ)         │ 最常用的类型             │
└──────────────┴────────────────┴─────────────────────────┘

💡 如何选择度量类型？

1. COSINE（推荐默认选择）
   ✓ 最常用，适用于大多数语义检索场景
   ✓ 不受向量长度影响，只关注方向
   ✓ Embedding 模型通常输出归一化向量

2. L2（欧几里得距离）
   ✓ 适用于向量未归一化的场景
   ✓ 考虑向量的绝对位置

3. IP（内积）
   ✓ 与 COSINE 在归一化后等价
   ✓ 某些场景下计算更快

📊

## 示例 6: Collection 操作大全

Collection 的常见操作：创建、查看、删除、重命名等。

In [20]:
def collection_operations():
    """
    Collection 的常见操作

    创建、查看、删除、重命名等
    """
    print("=" * 60)
    print("示例 6: Collection 操作大全")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)

    # 1. 列出所有 Collection
    print("1. 列出所有集合：")
    collections = client.list_collections()
    for col in collections:
        print(f"   - {col}")
    print()

    # 2. 检查 Collection 是否存在
    print("2. 检查集合是否存在：")
    test_name = "simple_docs"
    exists = client.has_collection(test_name)
    print(f"   {test_name}: {'存在' if exists else '不存在'}")
    print()

    # 3. 获取 Collection 详细信息
    if exists:
        print("3. 获取集合详细信息：")
        info = client.describe_collection(test_name)
        print(f"   名称：{info['collection_name']}")
        # 从 fields 中获取维度
        dimension = None
        for field in info.get('fields', []):
            if field.get('type') == 101:  # 101 表示向量类型
                dimension = field.get('params', {}).get('dim')
                break
        print(f"   维度：{dimension}")
        print(f"   度量类型：COSINE")
        print(f"   主键自增：{info['auto_id']}")
        print()

    # 4. 获取 Collection 行数
    if exists:
        print("4. 获取集合行数：")
        count = client.get_collection_stats(test_name)
        print(f"   行数：{count.get('row_count', 0)}")
        print()

    # 5. 删除 Collection（演示用，不实际执行）
    print("5. 删除集合（示例代码，不执行）：")
    print("   client.drop_collection('collection_name')")
    print()

    # 6. 重命名 Collection
    print("6. 重命名集合（示例代码，不执行）：")
    print("   client.rename_collection('old_name', 'new_name')")

In [21]:
# 运行示例 6
collection_operations()

示例 6: Collection 操作大全
1. 列出所有集合：
   - simple_docs
   - multi_vector_docs
   - custom_docs
   - dynamic_field_docs

2. 检查集合是否存在：
   simple_docs: 存在

3. 获取集合详细信息：
   名称：simple_docs
   维度：768
   度量类型：COSINE
   主键自增：True

4. 获取集合行数：
   行数：0

5. 删除集合（示例代码，不执行）：
   client.drop_collection('collection_name')

6. 重命名集合（示例代码，不执行）：
   client.rename_collection('old_name', 'new_name')


## 运行全部示例

In [ ]:
# 主程序入口 - 运行所有示例
if __name__ == '__main__':
    print("\n" + "=" * 70)
    print("  Milvus 基础 - 创建 Collection")
    print("  说明：学习如何在 Milvus 中创建集合")
    print("=" * 70 + "\n")

    print("【说明】")
    print("  本示例演示多种创建 Collection 的方式")
    print("  建表前先删表，确保代码可重复运行")
    print()

    # 运行示例
    create_simple_collection()
    print()

    create_custom_collection()
    print()

    create_multi_vector_collection()
    print()

    metric_types_explained()
    print()

    collection_operations()

    print()
    print("=" * 70)
    print("  Collection 学习完成！接下来学习：03_insert_data.py（插入数据）")
    print("=" * 70 + "\n")